In [8]:
# ==========================================
# CELDA 1: LIBRERÍAS Y CONFIGURACIÓN
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from fpdf import FPDF
import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ Celda 1: Librerías cargadas correctamente.")

✅ Celda 1: Librerías cargadas correctamente.


In [10]:
# ==========================================
# CELDA 2: CARGA DE DATOS Y GRAN MERGE
# ==========================================
print("🚀 INICIANDO CREACIÓN DEL DATAFRAME MAESTRO PARA MACHINE LEARNING...")

# 1. Carga de los 8 archivos
ruta_datos = '../data/clean/'

df_con = pd.read_csv(ruta_datos + 'conectividad_final_limpio.csv')
df_osm = pd.read_csv(ruta_datos + 'muni_station_osm_limpio.csv')
df_elec = pd.read_csv(ruta_datos + 'consumo_electrico_final_limpio.csv')
df_dem = pd.read_csv(ruta_datos + 'demografia_municipios_final.csv')
df_mig = pd.read_csv(ruta_datos + 'migracion_municipios_final_limpio.csv')
df_ren = pd.read_csv(ruta_datos + 'rentamedia_municipios_final_limpio.csv')
df_emp = pd.read_csv(ruta_datos + 'empresas_transporte_final_limpio.csv')
df_viirs = pd.read_csv(ruta_datos + 'viirsFinal_limpio.csv')

# 2. Transformación y unificación de claves
df_elec = df_elec.rename(columns={'Codigo': 'LAU_ID'})
df_mig = df_mig.rename(columns={'codigo_municipio': 'LAU_ID', 'anio': 'Anio', 'cantidad (personas)': 'migracion_total'})
df_ren = df_ren.rename(columns={'codigo_municipio': 'LAU_ID', 'anio': 'Anio'})
df_dem = df_dem.rename(columns={'year': 'Anio'})

df_viirs = df_viirs.rename(columns={'date': 'Anio'})
df_viirs['Anio'] = df_viirs['Anio'].astype(str).str[:4].astype(int)

df_con['municipio_norm'] = df_con['LAU_NAME'].str.lower().str.strip()
df_dem['municipio_norm'] = df_dem['municipio'].str.lower().str.strip()

# Conversión Wide-to-Long (Transporte)
df_emp_long = df_emp.melt(id_vars=['codigo', 'nombre', 'tipo'], var_name='Anio', value_name='num_empresas_transporte')
df_emp_long = df_emp_long.rename(columns={'codigo': 'LAU_ID'})
df_emp_long['Anio'] = df_emp_long['Anio'].astype(int) 

# 3. El Gran Merge
df_master_ml = pd.merge(df_con, df_osm, on='LAU_ID', how='left', suffixes=('', '_osm'))
df_master_ml = pd.merge(df_master_ml, df_elec, on='LAU_ID', how='left', suffixes=('', '_elec'))
df_master_ml = pd.merge(df_master_ml, df_mig[['LAU_ID', 'Anio', 'migracion_total']], on=['LAU_ID', 'Anio'], how='left')
df_master_ml = pd.merge(df_master_ml, df_ren[['LAU_ID', 'Anio', 'pib']], on=['LAU_ID', 'Anio'], how='left')
df_master_ml = pd.merge(df_master_ml, df_emp_long[['LAU_ID', 'Anio', 'num_empresas_transporte']], on=['LAU_ID', 'Anio'], how='left')
df_master_ml = pd.merge(df_master_ml, df_viirs[['LAU_ID', 'Anio', 'max', 'mean', 'min', 'stdDev', 'mean_prov']], on=['LAU_ID', 'Anio'], how='left')
df_master_ml = pd.merge(df_master_ml, df_dem, on=['municipio_norm', 'Anio'], how='left')
df_master_ml = df_master_ml.drop(columns=['municipio_norm', 'municipio', 'LAU_NAME_osm', 'index'], errors='ignore')

# 4. Feature Engineering
df_master_ml['pct_viviendas_vacias'] = np.where(df_master_ml['Viviendas totales'] > 0, df_master_ml['Viviendas vacías'] / df_master_ml['Viviendas totales'], 0)
df_master_ml['densidad_poblacion'] = np.where(df_master_ml['AREA_KM2'] > 0, df_master_ml['Total'] / df_master_ml['AREA_KM2'], 0)
df_master_ml['pct_bajo_consumo'] = np.where(df_master_ml['Viviendas totales'] > 0, df_master_ml['Viviendas con bajo consumo'] / df_master_ml['Viviendas totales'], 0)

print(f"✅ Dataframe Maestro creado. Dimensiones: {df_master_ml.shape}")

🚀 INICIANDO CREACIÓN DEL DATAFRAME MAESTRO PARA MACHINE LEARNING...
✅ Dataframe Maestro creado. Dimensiones: (666742, 81)


In [11]:
# ==========================================
# CELDA 3: IMPUTACIÓN Y PREPARACIÓN DE TARGETS
# ==========================================
print("🧹 INICIANDO LIMPIEZA Y PREPARACIÓN TEMPORAL...")

df_ml = df_master_ml.copy()

# 1. Imputación temporal y por ceros
columnas_temporales = ['pib', 'migracion_total', 'num_empresas_transporte', 'mean', 'max']
for col in columnas_temporales:
    if col in df_ml.columns:
        df_ml[col] = df_ml.groupby('LAU_ID')[col].transform(lambda x: x.ffill().bfill())

df_ml['migracion_total'] = df_ml['migracion_total'].fillna(0)
df_ml['num_empresas_transporte'] = df_ml['num_empresas_transporte'].fillna(0)
df_ml['pct_viviendas_vacias'] = df_ml['pct_viviendas_vacias'].fillna(0)
df_ml['pct_bajo_consumo'] = df_ml['pct_bajo_consumo'].fillna(0)

if 'pib' in df_ml.columns:
    df_ml['pib'] = df_ml['pib'].fillna(df_ml['pib'].median())

# 2. Variables (Features)
features = [
    'Total', 'Indice_Conectividad', 'Mediana consumo anual', 
    'pct_viviendas_vacias', 'mean_distance_km_to_station', 
    'pib', 'migracion_total', 'num_empresas_transporte', 'mean'
]

# 3. Target y División Train/Test
df_ml = df_ml.sort_values(by=['LAU_ID', 'Anio'])
df_ml['Poblacion_Futura'] = df_ml.groupby('LAU_ID')['Total'].shift(-1)
df_model = df_ml.dropna(subset=['Poblacion_Futura'] + features)

anio_corte = 2022
X_train = df_model[df_model['Anio'] < anio_corte][features]
y_train = df_model[df_model['Anio'] < anio_corte]['Poblacion_Futura']
X_test = df_model[df_model['Anio'] >= anio_corte][features]
y_test = df_model[df_model['Anio'] >= anio_corte]['Poblacion_Futura']

print(f"✅ Listos para entrenar. Pasado (Train): {len(X_train)} | Futuro (Test): {len(X_test)}")

🧹 INICIANDO LIMPIEZA Y PREPARACIÓN TEMPORAL...
✅ Listos para entrenar. Pasado (Train): 171941 | Futuro (Test): 76825


In [14]:
# ==========================================
# CELDA 4: ENTRENAMIENTO Y CÁLCULO DE RIESGO
# ==========================================
print("🤖 ENTRENANDO CEREBRO 1: MODELO NUMÉRICO...")
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_regressor.fit(X_train, y_train)
predicciones_num = rf_regressor.predict(X_test)

print("🚨 ENTRENANDO CEREBRO 2: RADAR DE PROBABILIDAD DE CAÍDA...")
y_train_class = (y_train < X_train['Total']).astype(int)
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
rf_classifier.fit(X_train, y_train_class)
probabilidades = rf_classifier.predict_proba(X_test)[:, 1] # Probabilidad matemática de caída

print("💎 CONSTRUYENDO EL ÍNDICE GEOLUMICA...")
df_resultados = X_test.copy()
df_resultados['Poblacion_Predicha_Futura'] = predicciones_num
df_resultados['Indice_Riesgo_GeoLumica'] = probabilidades * 100

def asignar_alerta(score):
    if score < 10: return '1. Sano / Sin Riesgo'
    elif score < 20: return '2. Riesgo Moderado'
    elif score < 35: return '3. Alerta Naranja (Vigilancia)'
    else: return '4. Riesgo Crítico de Vaciado'

df_resultados['Nivel_Alerta'] = df_resultados['Indice_Riesgo_GeoLumica'].apply(asignar_alerta)
print("✅ Motor predictivo terminado con éxito.")

🤖 ENTRENANDO CEREBRO 1: MODELO NUMÉRICO...
🚨 ENTRENANDO CEREBRO 2: RADAR DE PROBABILIDAD DE CAÍDA...
💎 CONSTRUYENDO EL ÍNDICE GEOLUMICA...
✅ Motor predictivo terminado con éxito.


In [15]:
# ==========================================
# CELDA 5: BUSINESS LOGIC Y EXPORTACIÓN PDF/CSV
# ==========================================
print("📦 PREPARANDO LOS ENTREGABLES PARA EL CLIENTE...")
hora_actual = datetime.datetime.now().strftime("%H%M")

# 1. Preparación de la base de datos final
df_final = df_resultados.copy()
df_final['LAU_NAME'] = df_model.loc[df_final.index, 'LAU_NAME']

df_export = df_final[['LAU_NAME', 'Total', 'Poblacion_Predicha_Futura', 'Indice_Riesgo_GeoLumica', 'Nivel_Alerta']].rename(columns={
    'LAU_NAME': 'Municipio', 'Total': 'Población 2025',
    'Poblacion_Predicha_Futura': 'Predicción 2026', 'Indice_Riesgo_GeoLumica': 'Score Riesgo (0-100)',
    'Nivel_Alerta': 'Estado de Alerta'
})

# Nos quedamos solo con la última predicción de cada municipio
df_export = df_export.drop_duplicates(subset=['Municipio'], keep='last')
df_export['Predicción 2026'] = df_export['Predicción 2026'].round(0).astype(int)
df_export['Score Riesgo (0-100)'] = df_export['Score Riesgo (0-100)'].round(1)

# 🔥 Lógica de Negocio: Si crece, apagamos el riesgo
condicion_crece = df_export['Predicción 2026'] >= df_export['Población 2025']
df_export.loc[condicion_crece, 'Estado de Alerta'] = '1. Sano / Sin Riesgo'
df_export.loc[condicion_crece, 'Score Riesgo (0-100)'] = 0.0
df_export = df_export.sort_values(by=['Score Riesgo (0-100)', 'Población 2025'], ascending=[False, True])

# 2. Exportar CSV Maestro
nombre_csv = f"GeoLumica_BaseDatos_2026_{hora_actual}.csv"
df_export.to_csv(nombre_csv, index=False, encoding='utf-8-sig')
print(f"✅ CSV exportado: {nombre_csv}")

# 3. Función generadora de PDFs
class PDFGeoLumica(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 15)
        self.set_text_color(44, 62, 80)
        self.cell(0, 10, 'GEOLUMICA - REPORTE EJECUTIVO DE RIESGO DEMOGRAFICO', 0, 1, 'C') 
        self.set_font('Arial', 'I', 11)
        self.set_text_color(127, 140, 141)
        self.cell(0, 10, f'Previsiones algoritmicas para el Ejercicio 2026', 0, 1, 'C')
        self.line(10, 30, 200, 30)
        self.ln(10)
    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.set_text_color(127, 140, 141)
        self.cell(0, 10, f'Pagina {self.page_no()}', 0, 0, 'C')

def generar_pdf(dataframe_top50, nombre_archivo, subtitulo):
    pdf = PDFGeoLumica()
    pdf.add_page()
    pdf.set_font('Arial', 'B', 12)
    pdf.set_text_color(44, 62, 80)
    pdf.cell(0, 10, subtitulo, 0, 1, 'L')
    pdf.set_font('Arial', '', 10)
    pdf.set_text_color(0, 0, 0)
    texto = ("El siguiente listado muestra los municipios identificados por el algoritmo predictivo "
             "de GeoLumica con mayor probabilidad de colapso demografico para el proximo ano.")
    pdf.multi_cell(0, 6, texto)
    pdf.ln(5)

    pdf.set_font('Arial', 'B', 9)
    pdf.set_fill_color(52, 73, 94)
    pdf.set_text_color(255, 255, 255)
    col_anchos = [55, 25, 30, 30, 50]
    cabeceras = ['Municipio', 'Pob. Actual', 'Prevision 2026', 'Score', 'Estado de Alerta']
    for i in range(len(cabeceras)):
        pdf.cell(col_anchos[i], 8, cabeceras[i], border=1, align='C', fill=True)
    pdf.ln()

    pdf.set_font('Arial', '', 8)
    pdf.set_text_color(0, 0, 0)
    for index, row in dataframe_top50.iterrows():
        pdf.set_fill_color(253, 237, 236) 
        muni = str(row['Municipio']).encode('latin-1', 'ignore').decode('latin-1')
        alerta = str(row['Estado de Alerta']).encode('latin-1', 'ignore').decode('latin-1')
        
        pdf.cell(col_anchos[0], 7, muni[:30], border=1, fill=True)
        pdf.cell(col_anchos[1], 7, f"{row['Población 2025']:,}", border=1, align='C', fill=True)
        pdf.cell(col_anchos[2], 7, f"{row['Predicción 2026']:,}", border=1, align='C', fill=True)
        pdf.set_font('Arial', 'B', 8)
        pdf.cell(col_anchos[3], 7, str(row['Score Riesgo (0-100)']), border=1, align='C', fill=True)
        pdf.set_font('Arial', '', 8)
        pdf.cell(col_anchos[4], 7, alerta, border=1, align='C', fill=True)
        pdf.ln()
    pdf.output(nombre_archivo)
    print(f"✅ PDF Generado: {nombre_archivo}")

# Exportar PDF 1 (General - Todas las ciudades)
generar_pdf(df_export.head(50), f"GeoLumica_Reporte_2026_General_{hora_actual}.pdf", 'Top 50 Municipios (General)')

# Exportar PDF 2 (Filtrado - España Vaciada < 20k habs)
df_pueblos = df_export[df_export['Población 2025'] <= 20000]
generar_pdf(df_pueblos.head(50), f"GeoLumica_Reporte_2026_Pueblos_{hora_actual}.pdf", 'Top 50 Riesgo Critico (< 20k habs)')

📦 PREPARANDO LOS ENTREGABLES PARA EL CLIENTE...
✅ CSV exportado: GeoLumica_BaseDatos_2026_1803.csv
✅ PDF Generado: GeoLumica_Reporte_2026_General_1803.pdf
✅ PDF Generado: GeoLumica_Reporte_2026_Pueblos_1803.pdf
